In [1]:
import geopandas as gpd
import  os 
import pandas as pd 
usuario = os.getlogin()


In [2]:

# yucatan = gpd.read_file(fr"C:\Users\{usuario}\Downloads\Anexo_1_yuc_validado.gpkg")

In [3]:
base = gpd.read_file(fr"C:\Users\{usuario}\Downloads\mapa_imo\mapa_primer_nivel\base\UPN_IMO_IMB_CASASSALUD_sin_CESSA.gpkg")

In [4]:
# sus = gpd.read_file(fr"C:\Users\{usuario}\Downloads\mapa_imo\mapa_primer_nivel\base\voronoi nuevo.gpkg")

In [6]:
consultorio = pd.read_excel(fr"C:\Users\{usuario}\Downloads\mapa_imo\mapa_primer_nivel\base\asignacion_definitiva_y_aprovechamiento.xlsx",sheet_name="Unidades",skiprows=2)

In [7]:
productividad =  pd.read_parquet(fr"C:\Users\{usuario}\Downloads\mapa_imo\mapa_primer_nivel\base\productividad_coplamar_imb.parquet")

In [8]:
unid = gpd.read_file(fr"C:\Users\{usuario}\Downloads\mapa_imo\mapa_primer_nivel\base\Unidades aceptadas_completo3.gpkg")
poligono = gpd.read_file(fr"C:\Users\{usuario}\Downloads\mapa_imo\mapa_primer_nivel\base\voronoi nuevo.gpkg")

In [9]:
# poligono = poligono.rename(columns={"CLUES":"CLUES_1"})

In [10]:
base

,clues,nombre_unidad,institucion,entidad,municipio,geometry
0,BCIMB000051,COLONIA LOMA LINDA,IMB,BAJA CALIFORNIA,ENSENADA,POINT (-116.6303 31.8823)
1,BCIMB000063,COLONIA POPULAR 2,IMB,BAJA CALIFORNIA,ENSENADA,POINT (-116.60868 31.89139)
2,BCIMB000075,COLONIA POPULAR 89,IMB,BAJA CALIFORNIA,ENSENADA,POINT (-116.57395 31.89444)
3,BCIMB000092,FRACCIONAMIENTO MAR,IMB,BAJA CALIFORNIA,ENSENADA,POINT (-116.56631 31.87835)
4,BCIMB000104,COL. JALISCO,IMB,BAJA CALIFORNIA,ENSENADA,POINT (-116.57674 31.85239)
...,...,...,...,...,...,...
15318,SCCSA003556,SAN RAMON,CSA,ZACATECAS,VILLA DE COS,POINT (-102.42933 23.15064)
15319,SCCSA003557,VICENTE GUERRERO,CSA,ZACATECAS,VILLA DE COS,POINT (-102.34906 23.95269)
15320,SCCSA003558,colonia diez de noviembre,CSA,ZACATECAS,VILLA GONZALEZ ORTEGA,POINT (-101.9344 22.61371)
15321,SCCSA003559,EL SALITRE,CSA,ZACATECAS,VILLA HIDALGO,POINT (-101.78083 22.40297)


In [11]:
poligono.columns = poligono.columns.str.lower().str.replace(' ', '_')

In [12]:
unid.columns = unid.columns.str.lower().str.replace(' ', '_')

In [13]:
consultorio.columns = consultorio.columns.str.lower().str.replace(' ', '_')
consultorio['clues'] = consultorio['clues'].astype(str).str.strip()
consultorio['_escenario_normalizado'] = (
    consultorio['escenario'].astype('string').str.strip().str.casefold()
)
consultorio['_prioridad'] = (
    consultorio['_escenario_normalizado']
    .map({
        'Después'.casefold(): 0,
        'Nuevas (casas de salud)'.casefold(): 1,
    })
    .fillna(2)
)
filas_consultorio_originales = len(consultorio)
consultorio = (
    consultorio
    .sort_values('_prioridad', kind='stable')
    .drop_duplicates(subset='clues', keep='first')
    .drop(columns=['_escenario_normalizado', '_prioridad'])
    .reset_index(drop=True)
)

print(f"Consultorios antes de priorizar: {filas_consultorio_originales:,}")
print(f"CLUES únicas después de priorizar: {len(consultorio):,}")

Consultorios antes de priorizar: 30,344
CLUES únicas después de priorizar: 15,439


In [16]:
poligono = poligono.rename(columns={'clues_final': 'clues'})

In [17]:
poligono.columns

Index(['fid', 'cvegeo', 'pobtot', 'pobfem', 'pobmas', 'p_0a2', 'pob0_14',
       'p_60ymas', 'p_60ymas_f', 'p_60ymas_m', 'p_15a49_f', 'psinder',
       'pder_ss', 'pder_imss', 'pder_segp', 'pder_imssb', 'pder_iste',
       'pder_istee', 'pafil_pdom', 'sin_der', 'entidad', 'clues_sin',
       'clues_con', 'se_reasigna', 'duracion_sin', 'duracion_con',
       'diferencia_duracion', 'distancia_sin', 'distancia_con',
       'diferencia_distancia', 'poblacion_total_2026',
       'poblacion_sin_derechohabiencia_2026', 'cambio_categorico', 'clues',
       'geometry'],
      dtype='object')

In [18]:
diagnostico_geometrico = {
    "poligono_crs": str(poligono.crs),
    "poligono_tipos": poligono.geom_type.value_counts().to_dict(),
    "poligono_registros": len(poligono),
    "poligono_clues_unicas": poligono["clues"].nunique(dropna=True),
    "poligono_geometrias_validas": int(poligono.geometry.is_valid.sum()),
    "unid_crs": str(unid.crs),
    "unid_tipos": unid.geom_type.value_counts().to_dict(),
    "unid_registros": len(unid),
    "unid_clues_unicas": unid["clues"].nunique(dropna=True),
}
diagnostico_geometrico

{'poligono_crs': 'EPSG:4326',
 'poligono_tipos': {'MultiPolygon': 15370},
 'poligono_registros': 15370,
 'poligono_clues_unicas': 15369,
 'poligono_geometrias_validas': 15370,
 'unid_crs': 'EPSG:4326',
 'unid_tipos': {'Point': 8600},
 'unid_registros': 8600,
 'unid_clues_unicas': 5041}

In [19]:
poligono_metrico = poligono.to_crs("EPSG:6372")
complejidad_original = int(poligono_metrico.geometry.get_coordinates().shape[0])
pruebas_simplificacion = {}

for tolerancia_m in (100, 250, 500):
    geometria_simplificada = poligono_metrico.geometry.simplify(
        tolerancia_m,
        preserve_topology=True,
    )
    pruebas_simplificacion[tolerancia_m] = {
        "coordenadas": int(geometria_simplificada.get_coordinates().shape[0]),
        "reduccion_pct": round(
            (1 - geometria_simplificada.get_coordinates().shape[0] / complejidad_original) * 100,
            1,
        ),
    }

{"coordenadas_originales": complejidad_original, "pruebas": pruebas_simplificacion}

{'coordenadas_originales': 936113,
 'pruebas': {100: {'coordenadas': 328935, 'reduccion_pct': 64.9},
  250: {'coordenadas': 246135, 'reduccion_pct': 73.7},
  500: {'coordenadas': 183085, 'reduccion_pct': 80.4}}}

In [20]:
import json
import zlib
from pathlib import Path

salida_voronoi = Path.cwd() / "public" / "voronoi"
salida_voronoi.mkdir(parents=True, exist_ok=True)

for archivo_anterior in salida_voronoi.glob("*.geojson"):
    archivo_anterior.unlink()

voronoi_web = poligono.loc[
    poligono["clues"].notna(),
    ["clues", "geometry"],
].copy()
voronoi_web["clues"] = voronoi_web["clues"].astype(str).str.strip()
voronoi_web["fragmento_id"] = voronoi_web["clues"].map(
    lambda clues: zlib.crc32(clues.encode("utf-8")) % 32
)
voronoi_web = voronoi_web.to_crs("EPSG:6372")
voronoi_web["geometry"] = voronoi_web.geometry.simplify(100, preserve_topology=True)
voronoi_web = voronoi_web.to_crs("EPSG:4326")

indice_voronoi = {}
for fragmento_id, fragmento in voronoi_web.groupby("fragmento_id", sort=True):
    nombre_archivo = f"fragmento-{fragmento_id:02d}.geojson"
    fragmento_salida = fragmento[["clues", "geometry"]].copy()
    (salida_voronoi / nombre_archivo).write_text(
        fragmento_salida.to_json(drop_id=True, separators=(",", ":")),
        encoding="utf-8",
    )
    indice_voronoi.update(dict.fromkeys(fragmento_salida["clues"], nombre_archivo))

(salida_voronoi / "index.json").write_text(
    json.dumps(indice_voronoi, ensure_ascii=False, separators=(",", ":")),
    encoding="utf-8",
)

tamanos_fragmentos = [archivo.stat().st_size for archivo in salida_voronoi.glob("*.geojson")]
tamano_mb = sum(archivo.stat().st_size for archivo in salida_voronoi.iterdir()) / 1024**2
print(f"Voronoi exportado: {len(voronoi_web):,} polígonos en {len(tamanos_fragmentos)} fragmentos")
print(f"Índice: {len(indice_voronoi):,} CLUES")
print(f"Tamaño total: {tamano_mb:.1f} MB")
print(f"Fragmento mayor: {max(tamanos_fragmentos) / 1024**2:.1f} MB")

Voronoi exportado: 15,369 polígonos en 32 fragmentos
Índice: 15,369 CLUES
Tamaño total: 14.4 MB
Fragmento mayor: 0.5 MB


In [21]:
for nombre, gdf in {"poligono": poligono, "unid": unid}.items():
    print(nombre)
    print("CRS:", gdf.crs)
    print(gdf.geom_type.value_counts())
    print("Registros:", len(gdf))

poligono
CRS: EPSG:4326
MultiPolygon    15370
Name: count, dtype: int64
Registros: 15370
unid
CRS: EPSG:4326
Point    8600
Name: count, dtype: int64
Registros: 8600


In [22]:
unid.columns

Index(['uid', 'id_temp_sus', 'id_temp_sheets_edos', 'operador',
       'institucion_que_propone_espacio', 'entidad', 'municipio', 'nombre',
       'clues', 'estrato', 'mod_id', 'modalidad', 'lat', 'lon',
       'origen_estrato', 'cluster_rural_id', 'cluster_rural_tamano_unidades',
       'regla_cluster', 'motivo_decision', 'estatus_final', 'num_consultorios',
       'fid', 'cvegeo', 'pobtot', 'pobfem', 'pobmas', 'p_0a2', 'pob0_14',
       'p_60ymas', 'p_60ymas_f', 'p_60ymas_m', 'p_15a49_f', 'psinder',
       'pder_ss', 'pder_imss', 'pder_segp', 'pder_imssb', 'pder_iste',
       'pder_istee', 'pafil_pdom', 'sin_der', 'nom_mun', 'dist_km', 'dur_min',
       'clues_2', 'institucion', 'nombre_unidad',
       'población_con_derechohabiencia_2026',
       'población_sin_derechohabiencia_2026', 'población_total_2026',
       'geometry'],
      dtype='object')

In [23]:
productividad

,clues,nombre_de_la_unidad,entidad,cve_ent,municipio,cve_mun,localidad,cve_loc,cvegeo,poblacion_con_derechohabiencia_2026,consulta_general
0,YNSSA014046,CENTRO DE SALUD DE MOTUL,YUCATAN,31,MOTUL,052,MOTUL DE CARRILLO PUERTO,0001,310520001,0.0,NaN
1,HGIMB000035,SAN BARTOLO,HIDALGO,13,ACATLAN,001,SAN BARTOLO (EL LLANO),0020,130010020,0.0,2251.0
2,SPIMB002516,CENTRO DE SALUD HUEHUETLAN,SAN LUIS POTOSI,24,HUEHUETLAN,018,HUEHUETLAN,0001,240180001,1.0,5824.0
3,YNSSA000705,CENTRO DE SALUD XOHUAYAN,YUCATAN,31,OXKUTZCAB,056,XOHUAYAN,0013,310560013,10.0,NaN
4,CMIMB000260,CENTRO DE SALUD CAMPO CUATRO,COLIMA,06,COMALA,003,CAMPO CUATRO,0008,060030008,11.0,807.0
...,...,...,...,...,...,...,...,...,...,...,...
11595,SRIMB002401,CENTRO DE SALUD URBANO NOGALES,SONORA,26,NOGALES,043,HEROICA NOGALES,0001,260430001,179373.0,17753.0
11596,YNSSA000466,CENTRO DE SALUD KANASÍN,YUCATAN,31,KANASIN,041,KANASIN,0001,310410001,190863.0,NaN
11597,QRIMB001430,CENTRO DE SALUD URBANO 01 PLAYA DEL CARMEN,QUINTANA ROO,23,SOLIDARIDAD,008,PLAYA DEL CARMEN,0001,230080001,182283.0,1900.0
11598,SLIMB003012,CENTRO DE SALUD MOCHIS II,SINALOA,25,AHOME,001,LOS MOCHIS,0001,250010001,193982.0,59650.0


In [24]:
productividad

,clues,nombre_de_la_unidad,entidad,cve_ent,municipio,cve_mun,localidad,cve_loc,cvegeo,poblacion_con_derechohabiencia_2026,consulta_general
0,YNSSA014046,CENTRO DE SALUD DE MOTUL,YUCATAN,31,MOTUL,052,MOTUL DE CARRILLO PUERTO,0001,310520001,0.0,NaN
1,HGIMB000035,SAN BARTOLO,HIDALGO,13,ACATLAN,001,SAN BARTOLO (EL LLANO),0020,130010020,0.0,2251.0
2,SPIMB002516,CENTRO DE SALUD HUEHUETLAN,SAN LUIS POTOSI,24,HUEHUETLAN,018,HUEHUETLAN,0001,240180001,1.0,5824.0
3,YNSSA000705,CENTRO DE SALUD XOHUAYAN,YUCATAN,31,OXKUTZCAB,056,XOHUAYAN,0013,310560013,10.0,NaN
4,CMIMB000260,CENTRO DE SALUD CAMPO CUATRO,COLIMA,06,COMALA,003,CAMPO CUATRO,0008,060030008,11.0,807.0
...,...,...,...,...,...,...,...,...,...,...,...
11595,SRIMB002401,CENTRO DE SALUD URBANO NOGALES,SONORA,26,NOGALES,043,HEROICA NOGALES,0001,260430001,179373.0,17753.0
11596,YNSSA000466,CENTRO DE SALUD KANASÍN,YUCATAN,31,KANASIN,041,KANASIN,0001,310410001,190863.0,NaN
11597,QRIMB001430,CENTRO DE SALUD URBANO 01 PLAYA DEL CARMEN,QUINTANA ROO,23,SOLIDARIDAD,008,PLAYA DEL CARMEN,0001,230080001,182283.0,1900.0
11598,SLIMB003012,CENTRO DE SALUD MOCHIS II,SINALOA,25,AHOME,001,LOS MOCHIS,0001,250010001,193982.0,59650.0


In [25]:
consultorio.columns


Index(['escenario', 'clues', 'tipo_de_unidad', 'localidades',
       'población_total_2026', 'población_sin_derechohabiencia_2026',
       'consultorios_habilitados', 'consultorios_inhabilitados',
       'total_consultorios', 'consultorios_usados',
       'población_por_consultorio', 'razón_de_aprovechamiento', 'categoría',
       'supuesto_1_consultorio'],
      dtype='object')

In [26]:
len(consultorio)

15439

In [27]:
len(base)

15323

In [28]:
base.columns

Index(['clues', 'nombre_unidad', 'institucion', 'entidad', 'municipio',
       'geometry'],
      dtype='object')

In [29]:
consultorio

,escenario,clues,tipo_de_unidad,localidades,población_total_2026,población_sin_derechohabiencia_2026,consultorios_habilitados,consultorios_inhabilitados,total_consultorios,consultorios_usados,población_por_consultorio,razón_de_aprovechamiento,categoría,supuesto_1_consultorio
0,Después,BCIMB000051,IMSS Bienestar,5,5730,2116,5.0,1.0,6.0,6,955.000000,0.318333,<0.50,No
1,Después,BCIMB000063,IMSS Bienestar,8,10546,3893,3.0,0.0,3.0,3,3515.333333,1.171778,0.80–1.20,No
2,Después,BCIMB000075,IMSS Bienestar,23,22289,8229,3.0,0.0,3.0,3,7429.666667,2.476556,>1.50,No
3,Después,BCIMB000080,IMSS Bienestar,17,26387,9739,7.0,0.0,7.0,7,3769.571429,1.256524,>1.20–1.50,No
4,Después,BCIMB000092,IMSS Bienestar,13,10683,3943,3.0,0.0,3.0,3,3561.000000,1.187000,0.80–1.20,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15434,Antes,YNSSA001154,IMSS Bienestar,46,30711,21144,2.0,0.0,2.0,2,15355.500000,5.118500,>1.50,No
15435,Antes,YNSSA001562,IMSS Bienestar,4,2074,1426,1.0,0.0,1.0,1,2074.000000,0.691333,0.50–<0.80,No
15436,Antes,YNSSA002535,IMSS Bienestar,5,1904,1282,3.0,0.0,3.0,3,634.666667,0.211556,<0.50,No
15437,Antes,YNSSA014080,IMSS Bienestar,6,3758,3415,1.0,0.0,1.0,1,3758.000000,1.252667,>1.20–1.50,No


In [30]:
base['clues'] = base['clues'].astype(str).str.strip()
base = (
    base
    .drop(columns=['total_consultorios', 'población_por_consultorio'], errors='ignore')
    .drop_duplicates(subset='clues', keep='first')
    .merge(
        consultorio[['clues', 'total_consultorios', 'población_por_consultorio']],
        on='clues',
        how='left',
        validate='one_to_one',
    )
)

print(f"Base después del merge de consultorios: {len(base):,} filas")

Base después del merge de consultorios: 15,323 filas


In [31]:
len(base)

15323

In [32]:
base = base.merge(
    productividad[['clues','consulta_general']],
    on='clues',
    how='left'
)

In [33]:
len(base)

15323

In [34]:
base.columns

Index(['clues', 'nombre_unidad', 'institucion', 'entidad', 'municipio',
       'geometry', 'total_consultorios', 'población_por_consultorio',
       'consulta_general'],
      dtype='object')

In [35]:
# base = base[base["nivel_atencion"]=="PRIMER NIVEL"]

In [36]:
# tipologias = [
#     'CENTROS DE SALUD CON SERVICIOS AMPLIADOS',
#     'UNIDAD MÓVIL',
#     'URBANO DE 01 NÚCLEOS BÁSICOS',
#     'URBANO DE 05 NÚCLEOS BÁSICOS',
#     'RURAL DE 01 NÚCLEO BÁSICO',
#     'URBANO DE 02 NÚCLEOS BÁSICOS',
#     'URBANO DE 11 NÚCLEOS BÁSICOS',
#     'URBANO DE 12 NÚCLEOS BÁSICOS Y MÁS',
#     'RURAL DE 02 NÚCLEOS BÁSICOS',
#     'URBANO DE 06 NÚCLEOS BÁSICOS',
#     'URBANO DE 03 NÚCLEOS BÁSICOS',
#     'URBANO DE 07 NÚCLEOS BÁSICOS',
#     'URBANO DE 08 NÚCLEOS BÁSICOS',
#     'URBANO DE 04 NÚCLEOS BÁSICOS'
# ]

# base = base[base['NOMBRE DE TIPOLOGIA'].isin(tipologias)]

In [37]:
conteo_imo = base["institucion"].unique()
conteo_imo

array(['IMB', 'IMS', 'CSA'], dtype=object)

In [38]:
base.columns

Index(['clues', 'nombre_unidad', 'institucion', 'entidad', 'municipio',
       'geometry', 'total_consultorios', 'población_por_consultorio',
       'consulta_general'],
      dtype='object')

In [39]:
len(base)

15323

In [40]:
from pathlib import Path

columnas_mapa = [
    'clues', 'nombre_unidad', 'institucion', 'entidad', 'municipio', 'geometry'
]

consultorio_mapa = consultorio.loc[:, [
    'clues', 'total_consultorios', 'población_por_consultorio'
]].copy()

productividad_mapa = productividad.loc[:, ['clues', 'consulta_general']].copy()
productividad_mapa['clues'] = productividad_mapa['clues'].astype(str).str.strip()

base_mapa = base.loc[
    base["institucion"].isin(["IMS", "IMB", "CSA"]),
    columnas_mapa,
].copy()

base_mapa["institucion"] = base_mapa["institucion"].replace({"IMS": "IMO"})
base_mapa['clues'] = base_mapa['clues'].astype(str).str.strip()
base_mapa = base_mapa.drop_duplicates(subset='clues', keep='first')
base_mapa = base_mapa.merge(consultorio_mapa, on='clues', how='left', validate='one_to_one')
base_mapa = base_mapa.merge(productividad_mapa, on='clues', how='left', validate='many_to_one')

salida_mapa = Path.cwd() / "public" / "mapa_base.geojson"
base_mapa.to_file(salida_mapa, driver="GeoJSON")

print(base_mapa["institucion"].value_counts().to_dict())
print(f"GeoJSON generado: {salida_mapa} ({len(base_mapa):,} CLUES)")
print(f"CLUES con consultorios: {base_mapa['total_consultorios'].notna().sum():,}")
print(f"CLUES con consulta general: {base_mapa['consulta_general'].notna().sum():,}")

{'IMB': 8077, 'CSA': 4012, 'IMO': 3234}
GeoJSON generado: c:\Users\jose.valdez\Downloads\mapa_imo\mapa_primer_nivel\public\mapa_base.geojson (15,323 CLUES)
CLUES con consultorios: 8,052
CLUES con consulta general: 11,010


In [41]:
escenario_normalizado = consultorio['escenario'].astype('string').str.strip().str.casefold()
consultorio_auditoria = consultorio.assign(
    _escenario_normalizado=escenario_normalizado,
    _prioridad=escenario_normalizado.map({
        'Después'.casefold(): 0,
        'Nuevas (casas de salud)'.casefold(): 1,
    }).fillna(2),
)
seleccion_auditoria = (
    consultorio_auditoria
    .sort_values('_prioridad', kind='stable')
    .drop_duplicates('clues', keep='first')
    .set_index('clues')
)
escenarios_por_clues = consultorio_auditoria.groupby('clues')['_escenario_normalizado'].agg(set)
esperados = escenarios_por_clues.map(
    lambda escenarios: (
        'Después'.casefold() if 'Después'.casefold() in escenarios
        else 'Nuevas (casas de salud)'.casefold()
        if 'Nuevas (casas de salud)'.casefold() in escenarios
        else None
    )
)
comparables = esperados.notna()
incumplimientos = seleccion_auditoria.loc[comparables, '_escenario_normalizado'] != esperados[comparables]

print(f"CLUES de consultorio revisadas: {len(seleccion_auditoria):,}")
print(f"Seleccionadas como Después: {(seleccion_auditoria['_escenario_normalizado'] == 'Después'.casefold()).sum():,}")
print(f"Seleccionadas como Nuevas (casas de salud): {(seleccion_auditoria['_escenario_normalizado'] == 'Nuevas (casas de salud)'.casefold()).sum():,}")
print(f"Incumplimientos de prioridad: {incumplimientos.sum():,}")
print(f"CLUES únicas exportadas: {base_mapa['clues'].nunique():,} de {len(base_mapa):,} filas")

CLUES de consultorio revisadas: 15,439
Seleccionadas como Después: 15,369
Seleccionadas como Nuevas (casas de salud): 0
Incumplimientos de prioridad: 0
CLUES únicas exportadas: 15,323 de 15,323 filas
